# Tarea: Ajuste de Datos por Mínimos Cuadrados

Este cuaderno documenta el proceso numérico para calcular el ajuste de mínimos cuadrados sobre un conjunto de 10 puntos experimentales. Se analizan cinco modelos distintos:
1. Grado 1 (Lineal)
2. Grado 2 (Cuadrático)
3. Grado 3 (Cúbico)
4. Semilogarítmico (Exponencial)
5. Logarítmico Doble (Potencial)

El objetivo es determinar la función $\hat{y} = f(x)$ que minimice el Error Cuadrático Medio ($\text{MSE}$).


In [ ]:
% --- 1. CONFIGURACIÓN E INICIALIZACIÓN DE DATOS ---
clear; clc; close all;

% Vectores de datos reales
x = [4.0, 4.2, 4.5, 4.7, 5.1, 5.5, 5.9, 6.3, 6.8, 7.1];
y = [102.56, 113.18, 130.11, 142.05, 167.53, 195.14, 224.87, 256.73, 299.50, 326.72];

% Número total de observaciones
m = length(x);

% Cálculo de sumatorias requeridas para los sistemas de ecuaciones
sum_x   = sum(x);         sum_y   = sum(y);
sum_x2  = sum(x.^2);     sum_xy  = sum(x.*y);
sum_x3  = sum(x.^3);     sum_x4  = sum(x.^4);
sum_x5  = sum(x.^5);     sum_x6  = sum(x.^6);
sum_x2y = sum((x.^2).*y);
sum_x3y = sum((x.^3).*y);


## a) Ajuste de Grado 1 (Lineal)

Se busca una ecuación de la forma:
$$y = a_1 x + a_0$$

Los coeficientes de la pendiente ($a_1$) e intersección ($a_0$) se obtienen directamente mediante las fórmulas explícitas de mínimos cuadrados derivadas de la condición de minimización $\frac{\partial E}{\partial a} = 0$.


In [ ]:
% --- AJUSTE LINEAL ---
a0 = (sum_y*sum_x2 - sum_xy*sum_x) / (m*sum_x2 - sum_x^2);
a1 = (m*sum_xy - sum_x*sum_y) / (m*sum_x2 - sum_x^2);

% Evaluamos la predicción del modelo
p1 = a1.*x + a0;

% Cálculo del Error Cuadrático Medio (MSE)
r1 = y - p1; % Residuos
MSE1 = sum(r1.^2) / m;

fprintf('Grado 1: y = %.4fx + (%.4f) | MSE = %.4f\n', a1, a0, MSE1);


## b) Ajuste de Grado 2 (Cuadrático)

Se busca una parábola de la forma:
$$y = a_2 x^2 + a_1 x + a_0$$

Se plantea el sistema matricial de ecuaciones normales $A_2 \cdot \vec{a} = B_2$:

$$\begin{bmatrix} m & \sum x & \sum x^2 \\ \sum x & \sum x^2 & \sum x^3 \\ \sum x^2 & \sum x^3 & \sum x^4 \end{bmatrix} \begin{bmatrix} a_0 \\ a_1 \\ a_2 \end{bmatrix} = \begin{bmatrix} \sum y \\ \sum xy \\ \sum x^2y \end{bmatrix}$$


In [ ]:
% --- AJUSTE CUADRÁTICO ---
A2 = [m,      sum_x,  sum_x2;
      sum_x,  sum_x2, sum_x3;
            sum_x2, sum_x3, sum_x4];
            B2 = [sum_y; sum_xy; sum_x2y];

            % Resolución del sistema numérico
            sol2 = linsolve(A2, B2);

            % Evaluación de la parábola: sol2(1)=a0, sol2(2)=a1, sol2(3)=a2
            p2 = sol2(1) + sol2(2).*x + sol2(3).*(x.^2);

            r2 = y - p2;
            MSE2 = sum(r2.^2) / m;

            fprintf('Grado 2: y = %.4fx^2 + (%.4f)x + (%.4f) | MSE = %.4f\n', sol2(3), sol2(2), sol2(1), MSE2);
            

## c) Ajuste de Grado 3 (Cúbico)

Se busca un polinomio cúbico de la forma:
$$y = a_3 x^3 + a_2 x^2 + a_1 x + a_0$$

Se expande la matriz de ecuaciones normales a un sistema lineal de $4 \times 4$.


In [ ]:
% --- AJUSTE CÚBICO ---
A3 = [m,      sum_x,  sum_x2, sum_x3;
      sum_x,  sum_x2, sum_x3, sum_x4;
            sum_x2, sum_x3, sum_x4, sum_x5;
                  sum_x3, sum_x4, sum_x5, sum_x6];
                  B3 = [sum_y; sum_xy; sum_x2y; sum_x3y];

                  sol3 = linsolve(A3, B3);

                  p3 = sol3(1) + sol3(2).*x + sol3(3).*(x.^2) + sol3(4).*(x.^3);

                  r3 = y - p3;
                  MSE3 = sum(r3.^2) / m;

                  fprintf('Grado 3: y = %.4fx^3 + (%.4f)x^2 + (%.4f)x + (%.4f) | MSE = %.4f\n', sol3(4), sol3(3), sol3(2), sol3(1), MSE3);
                  

## d) Ajuste Semilogarítmico (Exponencial)

Modela funciones del tipo:
$$y = a \cdot e^{bx}$$

Al aplicar logaritmo natural en ambos lados, transformamos la ecuación no lineal en una relación lineal:
$$\ln(y) = \ln(a) + bx$$

Se resuelve el sistema lineal sustituyendo $y$ por $\ln(y)$, y posteriormente se recupera la constante $a$ usando $a = e^{\ln(a)}$.


In [ ]:
% --- AJUSTE SEMILOG ---
ln_y = log(y);
sum_lny = sum(ln_y);
sum_xlny = sum(x.*ln_y);

A4 = [m,     sum_x;
      sum_x, sum_x2];
      B4 = [sum_lny; sum_xlny];

      sol4 = linsolve(A4, B4);

      % Transformación inversa con exp()
      p4 = exp(sol4(1) + sol4(2).*x);

      r4 = y - p4;
      MSE4 = sum(r4.^2) / m;

      fprintf('Semilog: y = %.4f * e^(%.4fx) | MSE = %.4f\n', exp(sol4(1)), sol4(2), MSE4);
      

## e) Ajuste Log-Log (Potencial)

Modela relaciones de potencia del tipo:
$$y = a \cdot x^b$$

Aplicando logaritmo natural a ambas variables ($x$ e $y$):
$$\ln(y) = \ln(a) + b \ln(x)$$

Esta doble transformación permite calcular los parámetros mediante mínimos cuadrados lineales estándar.


In [ ]:
% --- AJUSTE LOG-LOG ---
ln_x = log(x);
sum_lnx = sum(ln_x);
sum_lnx2 = sum(ln_x.^2);
sum_lnxlny = sum(ln_x.*ln_y);

A5 = [m,       sum_lnx;
      sum_lnx, sum_lnx2];
      B5 = [sum_lny; sum_lnxlny];

      sol5 = linsolve(A5, B5);

      % Transformación inversa
      p5 = exp(sol5(1) + sol5(2).*ln_x);

      r5 = y - p5;
      MSE5 = sum(r5.^2) / m;

      fprintf('Log-Log: y = %.4f * x^(%.4f) | MSE = %.4f\n', exp(sol5(1)), sol5(2), MSE5);
      

## Visualización de Gráficas y Comparación

Generamos la representación visual de los 5 modelos superpuestos sobre los datos reales para evaluar el comportamiento geométrico del ajuste.


In [ ]:
% --- GRAFICACIÓN ---
ps = linspace(min(x), max(x), 100);
int = interp1(x, y, ps, "spline");

figure;
scatter(x, y, 70, '*', 'DisplayName', 'Datos Reales');
hold on;
plot(ps, int, '-', 'LineWidth', 1.2, 'DisplayName', 'Interpolación Spline');
plot(x, p1, '--', 'LineWidth', 1.2, 'DisplayName', 'Grado 1 (Lineal)');
plot(x, p2, '--', 'LineWidth', 1.2, 'DisplayName', 'Grado 2 (Cuadrático)');
plot(x, p3, '-.', 'LineWidth', 1.2, 'DisplayName', 'Grado 3 (Cúbico)');
plot(x, p4, '--', 'LineWidth', 1.2, 'DisplayName', 'Semilog');
plot(x, p5, ':',  'LineWidth', 1.2, 'DisplayName', 'Log-Log');

grid on;
xlabel('Eje X');
ylabel('Eje Y');
title('Comparación de Modelos por Mínimos Cuadrados');
legend('Location', 'southeast');

% Tabla resumen de errores en consola
fprintf('\n=========================================\n');
fprintf('    TABLA COMPARATIVA DE ERROR (MSE)     \n');
fprintf('=========================================\n');
fprintf('1. Grado 1 (Lineal):     MSE = %.4f\n', MSE1);
fprintf('2. Grado 2 (Cuadrático): MSE = %.4f\n', MSE2);
fprintf('3. Grado 3 (Cúbico):     MSE = %.4f\n', MSE3);
fprintf('4. Semilog (Exponencial):MSE = %.4f\n', MSE4);
fprintf('5. Log-Log (Potencial):  MSE = %.4f\n', MSE5);
fprintf('=========================================\n');


## Conclusión y Selección del Mejor Ajuste

Analizando los valores obtenidos de Error Cuadrático Medio ($\text{MSE}$):

* **Ajuste Lineal y Semilog:** Muestran errores elevados ($\text{MSE} = 32.9013$ y $\text{MSE} = 41.7691$ respectivamente), lo cual indica que la tendencia de los datos no es rectilínea ni exponencial pura.
* **Ajuste Log-Log:** Reduce significativamente el error ($\text{MSE} = 0.0007$) y nos revela un exponente $b \approx 2.0195$, lo que apunta a un comportamiento parabólico subyacente.
* **Polinomio de Grado 2:** Es el **modelo óptimo**. Alcanza un error insignificante ($\text{MSE} \approx 0.0001$) ofreciendo el mejor balance entre precisión y simplicidad matemática.
* **Polinomio de Grado 3:** Aunque presenta un $\text{MSE}$ numéricamente idéntico o ligeramente menor ($\approx 0.00005$), el coeficiente cúbico $a_3 = -0.0137$ es despreciable. Agregar un grado extra añade complejidad sin aportar una mejora real en el ajuste.
